# Batch MS vegetation-index extraction

## In short

This notebook calculates vegetation indices for each plot in a field trial, for each flight date. It uses multispectral drone images (red, green, red edge, and near-infrared bands).

You give it:

- One multispectral orthomosaic (a stitched drone image) for each date.
- One plot grid shapefile (the plot boundaries) for each date. The notebook uses the plot grid from the RGB folder of the same date.

You get one table (CSV file) for each date. Each row is one plot. The columns show the mean value of the 4 bands and 46 vegetation indices for that plot.

To use it, set two folder paths at the end of the code cell. Then run the cell.

## Details

**Input folder layout**

```
site_folder/
  MS/
    YYYY-MM-DD/
      <orthomosaic>.tif      band 1 = Red, 2 = Green, 3 = NIR, 4 = RedEdge, 5 = Alpha (not used)
  RGB/
    YYYY-MM-DD/
      <plot_grid>.shp        same date as the MS folder, can be in a subfolder
```

The notebook looks for the plot grid in `RGB/<same date>/` first. If it finds no grid there, it looks in `MS/<date>/`.

**Output**

- One CSV for each date: `output_dir/<site>_<YYYY-MM-DD>_MS_VI.csv`. `<site>` is the name of `site_folder`.
- `<site>_MS_VI_batch_summary.csv`: the status, the raster, and the shapefile for each date.

**How to run**

1. In the `USER SETTINGS` block at the end of the code cell, set `site_folder` and `output_dir`.
2. To process only some dates, set `dates_to_process` (for example `["2024-11-20"]`). This step is optional.
3. Run the cell.
4. Open the summary CSV. Look for dates with a status other than `processed`.
5. To process more than one site, give a list of `run_batch_ms_vi` arguments to `run_batch_jobs`. This step is optional.

**Notes**

- The band order, the formulas, and the masking are the same as in `Original_Template_Script/Generate_MS indices_yam.ipynb`. This notebook does not need that file to run.
- A date folder can contain more than one `.tif` or `.shp`. In that case, `tif_sort_key` and `shapefile_sort_key` select one file. The summary CSV shows the files that the notebook used.
- The plot mean uses `rasterio.mask(..., filled=True, nodata=0)` and `np.nanmean`. Pixels inside the plot bounding box but outside the polygon get the value 0. The mean includes these zeros.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable, Optional, Union

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio import mask

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover - only used if tqdm is unavailable
    def tqdm(iterable, **kwargs):
        return iterable


# ============================================================
# Batch MS vegetation-index extraction
# Built to follow the same batch pattern as Batch_generate_rgb_indicies.ipynb,
# while keeping the MS band order and index formulas from
# Original_Template_Script/Generate_MS indices_yam.ipynb.
# ============================================================

DATE_RE = re.compile(r"^(\d{4}-\d{2}-\d{2}|\d{2}-\d{2}-\d{2})$")
EXTRACTION_LOGIC = "original_ms_notebook_batch_extraction"
PLOT_ID_INDEX = 0

DEFAULT_PLOT_ID_CANDIDATES = (
    "var",
    "id",
    "plot_id",
    "Plot_ID",
    "plot",
    "Plot",
    "name",
    "Name",
)

MS_INDEX_COLUMNS = [
    "Mean Green",
    "Mean Red",
    "Mean RedEdge",
    "Mean NIR",
    "NDVI",
    "gNDVI",
    "RVI_1",
    "GCI",
    "RGVI",
    "DVI",
    "SAVI",
    "MSAVI",
    "OSAVI",
    "RDVI",
    "TVI",
    "TSAVI",
    "PVI",
    "SAVI_2",
    "ATSAVI",
    "NDWI",
    "RVI_2",
    "MCARI",
    "MCARI_1",
    "MCARI_2",
    "MTVI_1",
    "MTVI_2",
    "R_MCARI_MTVI_2",
    "DATT",
    "NDCI",
    "PSRI",
    "PSRI2",
    "PSRI3",
    "SPVI",
    "TCARI",
    "R_TCARI_OSAVI",
    "RERI",
    "NDRE",
    "MTCI",
    "EVI_2",
    "RECI",
    "NGRDI",
    "ARI_2",
    "CRI_2",
    "NRVI",
    "OSAVI_2",
    "MEVI",
    "TVI_2",
    "MCARI_OSAVI",
    "TCARI_OSAVI",
    "WDRVI",
]


def safe_name(value, keep_spaces: bool = False, max_len: int = 120) -> str:
    """Return a filesystem-safe name while keeping labels readable."""
    text = str(value).strip()

    if keep_spaces:
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r'[\\/:*?"<>|]+', "", text)
    else:
        text = text.replace(" ", "_")
        text = re.sub(r"[^A-Za-z0-9_\-()]+", "", text)

    text = text.strip(" ._")
    return (text[:max_len] if len(text) > max_len else text) or "unknown"


def find_orthophotos_root(path: Union[Path, str]) -> Path:
    """Find the Orthophotos folder from a site/script/notebook path."""
    path = Path(path).expanduser()
    try:
        resolved = path.resolve()
    except FileNotFoundError:
        resolved = path.absolute()

    for candidate in [resolved, *resolved.parents]:
        if candidate.name == "Orthophotos":
            return candidate

    for candidate in [resolved, *resolved.parents]:
        if (candidate / "Photos").exists() and (candidate / "Code").exists():
            return candidate

    return Path.cwd().resolve()


def find_ms_folder(site_folder: Path) -> Path:
    """Find the MS folder for a site."""
    direct = site_folder / "MS"
    if direct.exists() and direct.is_dir():
        return direct

    # Also allow the user to pass the MS folder itself.
    if site_folder.name.upper() == "MS" and site_folder.exists() and site_folder.is_dir():
        return site_folder

    candidates = sorted(
        p for p in site_folder.rglob("MS")
        if p.is_dir() and "Clipped" not in p.parts
    )

    if not candidates:
        raise FileNotFoundError(f"No MS folder found under: {site_folder}")

    if len(candidates) > 1:
        preview = "\n".join(f"  {p}" for p in candidates[:10])
        raise ValueError(
            "More than one MS folder was found. Please set site_folder to one site folder, "
            "or set ms_folder directly.\n"
            f"Candidates:\n{preview}"
        )

    return candidates[0]


def collect_date_folders(
    ms_folder: Path,
    dates_to_process: Optional[Iterable[str]] = None,
) -> dict[str, Path]:
    date_folders = {
        p.name: p
        for p in ms_folder.iterdir()
        if p.is_dir() and DATE_RE.match(p.name)
    }

    if dates_to_process:
        requested = set(dates_to_process)
        date_folders = {d: p for d, p in date_folders.items() if d in requested}

    return dict(sorted(date_folders.items()))


def tif_sort_key(path: Path):
    """Rank .tif files in a date folder. The lowest score is used."""
    name = path.name.lower()
    score = 100

    if "reflectance" in name:
        score -= 40
    if "ms" in name or "multi" in name:
        score -= 20
    if "orthomosaic" in name or "orthophoto" in name:
        score -= 10
    if "alpha" in name:
        score += 30
    if "thumbnail" in name or "preview" in name:
        score += 80
    if "alias" in name:
        score += 100

    return score, name


def find_ms_tif_for_date(date_dir: Path) -> Optional[Path]:
    tif_files = sorted(
        (
            p for p in date_dir.iterdir()
            if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
        ),
        key=tif_sort_key,
    )
    return tif_files[0] if tif_files else None


def shapefile_sort_key(path: Path, date_dir: Path):
    """
    Prefer plot-grid shapefiles for a date folder.

    Supports both layouts:
        RGB/<date>/<date>-Go2_grid.shp
        RGB/<date>/<date>_GRID/<date>_GRID.shp

    Lower score is preferred. The recursive search happens in
    find_grid_shapefile_for_date(); this sort key just chooses the
    best .shp if several exist.
    """
    relative = str(path.relative_to(date_dir)).lower()
    name = path.name.lower()
    parent_name = path.parent.name.lower()
    score = 0

    # Strongly prefer files/folders that are clearly plot grids.
    if "grid" in name:
        score -= 50
    if "grid" in parent_name:
        score -= 30
    if "grid" not in relative:
        score += 100

    # Avoid unrelated/problem shapefiles when present.
    if "bad" in relative:
        score += 100
    if "boundary" in relative or "outline" in relative:
        score += 40

    # Prefer a .shp directly in the date folder over deeply nested files
    # only when the grid signal is otherwise tied.
    depth = len(path.relative_to(date_dir).parts)

    return score, depth, relative


def shapefile_has_field(path: Path, field: str) -> bool:
    try:
        columns = gpd.read_file(path, rows=1).columns
    except TypeError:
        columns = gpd.read_file(path).columns
    except Exception:
        return False

    return field in columns


def find_grid_shapefile_for_date(
    date_dir: Path,
    plot_id_field: Optional[str] = None,
) -> Optional[Path]:
    """
    Recursively find the best grid shapefile inside one date folder.

    This covers both common layouts:
        RGB/2025-06-03/06-03-25-Go2_grid.shp
        RGB/2024-07-24/07-24-24-RA_GRID/07-24-24-RA_GRID.shp
    """
    candidates = sorted(
        (p for p in date_dir.rglob("*.shp") if not p.name.startswith(".")),
        key=lambda p: shapefile_sort_key(p, date_dir),
    )

    if not candidates:
        return None

    if plot_id_field:
        for candidate in candidates:
            if shapefile_has_field(candidate, plot_id_field):
                return candidate

    return candidates[0]



def find_rgb_folder(site_folder: Path) -> Optional[Path]:
    """Find the RGB folder for a site, if present."""
    direct = site_folder / "RGB"
    if direct.exists() and direct.is_dir():
        return direct

    candidates = sorted(
        p for p in site_folder.rglob("RGB")
        if p.is_dir() and "Clipped" not in p.parts
    )

    if not candidates:
        return None

    if len(candidates) > 1:
        preview = "\n".join(f"  {p}" for p in candidates[:10])
        raise ValueError(
            "More than one RGB folder was found. Please set site_folder to one site folder.\n"
            f"Candidates:\n{preview}"
        )

    return candidates[0]


def find_grid_shapefile_for_ms_date(
    ms_date_dir: Path,
    site_folder: Path,
    ms_folder: Path,
    plot_id_field: Optional[str] = None,
) -> Optional[Path]:
    """
    Find the plot-grid shapefile for an MS date.

    The MS orthos live in MS/<date>, but the plot grids are the same
    shapefiles used by the RGB extraction. Therefore the preferred lookup is:

        1. RGB/<same date>/.../*.shp
        2. MS/<same date>/.../*.shp, only as a fallback
        3. Any other same-date folder under the site, as a final fallback
    """
    date_str = ms_date_dir.name
    search_dirs: list[Path] = []

    def add_search_dir(path: Optional[Path]) -> None:
        if path is None:
            return
        try:
            resolved = path.resolve()
        except FileNotFoundError:
            return
        if resolved.exists() and resolved.is_dir() and resolved not in search_dirs:
            search_dirs.append(resolved)

    # Primary source of plot grids for MS: the matching RGB date folder.
    rgb_folder = find_rgb_folder(site_folder)
    add_search_dir(rgb_folder / date_str if rgb_folder is not None else None)

    # Fallbacks for unusual folder layouts.
    add_search_dir(ms_date_dir)
    add_search_dir(site_folder / date_str)

    for candidate_dir in sorted(site_folder.rglob(date_str)):
        if not candidate_dir.is_dir():
            continue
        try:
            candidate_resolved = candidate_dir.resolve()
        except FileNotFoundError:
            continue
        if "extracted VI" in candidate_resolved.parts:
            continue
        add_search_dir(candidate_resolved)

    for search_dir in search_dirs:
        shapefile = find_grid_shapefile_for_date(
            search_dir,
            plot_id_field=plot_id_field,
        )
        if shapefile is not None:
            if rgb_folder is not None and search_dir == (rgb_folder / date_str).resolve():
                print(f"Using RGB shapefile for MS date {date_str}: {shapefile}")
            elif search_dir != ms_date_dir.resolve():
                print(f"Using shapefile from matching date folder: {search_dir}")
            return shapefile

    return None


def resolve_plot_id_field(
    gdf: gpd.GeoDataFrame,
    requested_field: Optional[str] = None,
    candidates: Iterable[str] = DEFAULT_PLOT_ID_CANDIDATES,
) -> str:
    if requested_field:
        if requested_field not in gdf.columns:
            raise ValueError(
                f"Plot ID field '{requested_field}' not found. "
                f"Available fields: {list(gdf.columns)}"
            )
        return requested_field

    for candidate in candidates:
        if candidate in gdf.columns:
            return candidate

    non_geometry_columns = [c for c in gdf.columns if c != gdf.geometry.name]
    if not non_geometry_columns:
        raise ValueError("No non-geometry fields found for plot IDs.")

    return non_geometry_columns[PLOT_ID_INDEX]


def extract_by_mask(src, shape_geom):
    """
    Extract the masked image based on the given shape geometry.

    MS band order from the original MS notebook:
      Band 1 = Red
      Band 2 = Green
      Band 3 = NIR
      Band 4 = RedEdge
      Band 5 = Alpha (ignored)
    """
    # Same as the original MS notebook: filled=True with nodata=0 keeps
    # zero-filled pixels outside the polygon in the cropped window, and
    # those zeros are included in np.nanmean.
    mask_img = mask.mask(dataset=src, shapes=[shape_geom], nodata=0, filled=True, crop=True)[0]
    mask_img = mask_img.astype(np.float32)

    red_mean = np.nanmean(mask_img[0, :, :])
    green_mean = np.nanmean(mask_img[1, :, :])
    nir_mean = np.nanmean(mask_img[2, :, :])
    rededge_mean = np.nanmean(mask_img[3, :, :])

    with np.errstate(divide="ignore", invalid="ignore"):
        # NDVI
        ndvi = (nir_mean - red_mean) / (nir_mean + red_mean)

        # gNDVI
        gndvi = (nir_mean - green_mean) / (nir_mean + green_mean)

        # RVI_1
        rvi_1 = nir_mean / red_mean

        # GCI
        gci = (nir_mean / green_mean) - 1.0

        # RGVI
        rgvi = red_mean / green_mean

        # DVI
        dvi = nir_mean - red_mean

        # SAVI
        L = 0.5
        savi = ((nir_mean - red_mean) / (nir_mean + red_mean + L)) * (1.0 + L)

        # MSAVI
        msavi = 0.5 * (
            (2.0 * nir_mean) + 1.0
            - np.sqrt(np.square(2.0 * nir_mean + 1.0) - 8.0 * (nir_mean - red_mean))
        )

        # OSAVI
        osavi = (nir_mean - red_mean) / (nir_mean + red_mean + 0.16)

        # RDVI
        rdvi = np.sqrt(np.square(nir_mean - red_mean) / (nir_mean + red_mean))

        # TVI
        tvi = 60.0 * (nir_mean - green_mean) - 100.0 * (red_mean - green_mean)

        # TSAVI
        a, b = 0.96916, 0.084726
        tsavi = (a * (nir_mean - a * red_mean - b)) / (a * nir_mean + red_mean - a * b)

        # PVI
        a, b = 0.96916, 0.084726
        pvi = (nir_mean - a * red_mean - b) / np.sqrt(1 + np.square(a))

        # SAVI_2
        a, b = 0.96916, 0.084726
        savi_2 = nir_mean / (red_mean - (b / a))

        # ATSAVI
        a, b, x = 0.96916, 0.084726, 0.08
        atsavi = (a * (-a * red_mean - b)) / (a * nir_mean + red_mean - a * b + x * (1 + np.square(a)))

        # NDWI
        ndwi = (green_mean - nir_mean) / (green_mean + nir_mean)

        # RVI_2
        rvi_2 = nir_mean / green_mean

        # MCARI
        mcari = (rededge_mean - red_mean - 0.2 * (rededge_mean - green_mean)) * (rededge_mean / red_mean)

        # MCARI_1
        mcari_1 = 1.2 * (2.5 * (nir_mean - red_mean) - 1.3 * (nir_mean - green_mean))

        # MCARI_2
        mcari_2 = (
            1.5 * (2.5 * (nir_mean - red_mean) - 1.3 * (nir_mean - green_mean))
            * (np.square(2.0 * nir_mean + 1))
            - (6.0 * nir_mean - 5.0 * red_mean)
            - 0.5
        )

        # MTVI_1
        mtvi_1 = 1.2 * (1.2 * (nir_mean - green_mean) - 2.5 * (red_mean - green_mean))

        # MTVI_2
        mtvi_2 = (
            1.5 * (1.2 * (nir_mean - green_mean) - 2.5 * (red_mean - green_mean))
            * (np.square(2 * nir_mean + 1))
            - (6.0 * nir_mean - 5.0 * red_mean)
            - 0.5
        )

        # R_MCARI_MTVI_2
        r_mcari_mtvi2 = (
            ((rededge_mean - red_mean - 0.2 * (rededge_mean - green_mean)) * (rededge_mean / red_mean))
            / (
                (1.5 * (1.2 * nir_mean - green_mean) - 2.5 * (red_mean - green_mean))
                * (np.square(2 * nir_mean + 1) - (6.0 * nir_mean - 5.0 * red_mean) - 0.5)
            )
        )

        # DATT
        datt = (nir_mean - rededge_mean) / (nir_mean - red_mean)

        # NDCI
        ndci = (rededge_mean - green_mean) / (rededge_mean + green_mean)

        # PSRI
        psri = (red_mean - green_mean) / rededge_mean

        # PSRI2
        psri2 = (red_mean - green_mean) / nir_mean

        # PSRI3
        psri3 = (rededge_mean - red_mean) / nir_mean

        # SPVI
        spvi = 0.4 * 3.7 * (nir_mean - red_mean) - 1.2 * np.absolute(green_mean - red_mean)

        # TCARI
        tcari = 3.0 * ((rededge_mean - red_mean) - 0.2 * (rededge_mean - green_mean) * (rededge_mean / red_mean))

        # R_TCARI_OSAVI
        r_tcari_osavi = (
            3.0 * ((rededge_mean - red_mean) - 0.2 * (rededge_mean - green_mean) * (rededge_mean / red_mean))
        ) / ((nir_mean - red_mean) / (nir_mean + red_mean + 0.16))

        # RERI
        reri = (rededge_mean - red_mean) / nir_mean

        # NDRE
        ndre = (nir_mean - rededge_mean) / (nir_mean + rededge_mean)

        # MTCI
        mtci = (nir_mean - rededge_mean) / (rededge_mean - red_mean)

        # EVI_2
        evi_2 = 2.5 * ((nir_mean - red_mean) / (nir_mean + 2.4 * red_mean + 1.0))

        # RECI
        reci = (nir_mean / rededge_mean) - 1

        # NGRDI
        ngrdi = (green_mean - red_mean) / (green_mean + red_mean)

        # ARI_2
        ari_2 = nir_mean * ((1.0 / green_mean) - (1.0 / rededge_mean))

        # CRI_2
        cri_2 = (1.0 / green_mean) - (1.0 / rededge_mean)

        # NRVI
        rvi = nir_mean / red_mean
        nrvi = (1 / rvi - 1) / (1 / rvi + 1)

        # OSAVI_2
        osavi_2 = ((1 + 0.16) * (nir_mean - red_mean)) / (nir_mean + red_mean + 0.16)

        # MEVI
        mevi = (2.5 * (nir_mean - red_mean)) / (nir_mean + 2.4 * red_mean + 1)

        # TVI_2
        ndvi_2 = (nir_mean - red_mean) / (nir_mean + red_mean)
        tvi_2 = np.sqrt(ndvi_2 + 0.5)

        # MCARI_OSAVI
        mcari_4 = (rededge_mean - red_mean - 0.2 * (rededge_mean - green_mean)) * (rededge_mean / red_mean)
        osavi_tmp = ((1 + 0.16) * (nir_mean - red_mean)) / (nir_mean + red_mean + 0.16)
        mcari_osavi = mcari_4 / osavi_tmp

        # TCARI_OSAVI
        tcari_4 = 3.0 * ((rededge_mean - red_mean) - 0.2 * (rededge_mean - green_mean) * (rededge_mean / red_mean))
        osavi_tmp2 = ((1 + 0.16) * (nir_mean - red_mean)) / (nir_mean + red_mean + 0.16)
        tcari_osavi = tcari_4 / osavi_tmp2

        # WDRVI
        L = 0.12
        wdrvi = (L * nir_mean - red_mean) / (L * nir_mean + red_mean)

    return {
        "Mean Green": green_mean,
        "Mean Red": red_mean,
        "Mean RedEdge": rededge_mean,
        "Mean NIR": nir_mean,
        "NDVI": ndvi,
        "gNDVI": gndvi,
        "RVI_1": rvi_1,
        "GCI": gci,
        "RGVI": rgvi,
        "DVI": dvi,
        "SAVI": savi,
        "MSAVI": msavi,
        "OSAVI": osavi,
        "RDVI": rdvi,
        "TVI": tvi,
        "TSAVI": tsavi,
        "PVI": pvi,
        "SAVI_2": savi_2,
        "ATSAVI": atsavi,
        "NDWI": ndwi,
        "RVI_2": rvi_2,
        "MCARI": mcari,
        "MCARI_1": mcari_1,
        "MCARI_2": mcari_2,
        "MTVI_1": mtvi_1,
        "MTVI_2": mtvi_2,
        "R_MCARI_MTVI_2": r_mcari_mtvi2,
        "DATT": datt,
        "NDCI": ndci,
        "PSRI": psri,
        "PSRI2": psri2,
        "PSRI3": psri3,
        "SPVI": spvi,
        "TCARI": tcari,
        "R_TCARI_OSAVI": r_tcari_osavi,
        "RERI": reri,
        "NDRE": ndre,
        "MTCI": mtci,
        "EVI_2": evi_2,
        "RECI": reci,
        "NGRDI": ngrdi,
        "ARI_2": ari_2,
        "CRI_2": cri_2,
        "NRVI": nrvi,
        "OSAVI_2": osavi_2,
        "MEVI": mevi,
        "TVI_2": tvi_2,
        "MCARI_OSAVI": mcari_osavi,
        "TCARI_OSAVI": tcari_osavi,
        "WDRVI": wdrvi,
    }


def extract_ms_indices_for_date(
    raster_path: Path,
    shapefile_path: Path,
    output_csv: Path,
    site_name: str,
    date_str: str,
    plot_id_field: Optional[str] = None,
    overwrite: bool = True,
) -> dict:
    if output_csv.exists() and not overwrite:
        return {
            "status": "skipped_existing_output",
            "date": date_str,
            "features": 0,
            "warnings": 0,
            "plot_id_field": "",
            "output_csv": str(output_csv),
        }

    plots = gpd.read_file(shapefile_path)
    if plots.empty:
        raise ValueError(f"Shapefile has no features: {shapefile_path}")

    plots = plots[plots.geometry.notnull()].copy()
    if plots.empty:
        raise ValueError(f"Shapefile has no valid geometries: {shapefile_path}")

    with rasterio.open(raster_path) as src:
        print("num bands:", src.count)
        print("band descriptions:", src.descriptions)

        if src.count < 4:
            raise ValueError(f"Expected at least 4 MS raster bands, found {src.count}: {raster_path}")

        if plots.crs and src.crs and plots.crs != src.crs:
            plots = plots.to_crs(src.crs)
        elif plots.crs is None:
            print(f"[WARNING] Shapefile CRS is missing: {shapefile_path}")
        elif src.crs is None:
            print(f"[WARNING] Raster CRS is missing: {raster_path}")

        resolved_plot_id_field = resolve_plot_id_field(plots, plot_id_field)

        records = []
        warnings_count = 0

        iterator = tqdm(
            plots.iterrows(),
            total=len(plots),
            desc=f"Extracting {date_str}",
        )

        for _, row in iterator:
            try:
                indices = extract_by_mask(src, row.geometry)
            except Exception as exc:
                warnings_count += 1
                plot_id = row[resolved_plot_id_field] if resolved_plot_id_field in row else "unknown"
                print(f"[WARNING] {date_str} plot {plot_id} failed: {exc}")
                indices = {column: np.nan for column in MS_INDEX_COLUMNS}

            records.append(indices)

    indices_df = pd.DataFrame(records, columns=MS_INDEX_COLUMNS)

    # Keep one row per polygon, matching the original MS notebook output structure.
    result = pd.concat(
        [plots.reset_index(drop=True), indices_df.reset_index(drop=True)],
        axis=1,
    )

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_csv, index=False, header=True)

    return {
        "status": "processed",
        "date": date_str,
        "features": len(result),
        "warnings": warnings_count,
        "plot_id_field": resolved_plot_id_field,
        "output_csv": str(output_csv),
    }



def infer_season_name_from_site_folder(site_folder: Union[Path, str]) -> Optional[str]:
    """Infer season folder name like 2024-25 or 2025-26 from the site path."""
    path = Path(site_folder)
    for part in [path.name, *[p.name for p in path.parents]]:
        match = re.search(r"(\d{4}-\d{2})", part)
        if match:
            return match.group(1)
    return None

def run_batch_ms_vi(
    site_folder: Optional[Union[Path, str]] = None,
    ms_folder: Optional[Union[Path, str]] = None,
    dates_to_process: Optional[Iterable[str]] = None,
    output_dir: Optional[Union[Path, str]] = None,
    output_root: Optional[Union[Path, str]] = None,
    season_name: Optional[str] = None,
    plot_id_field: Optional[str] = None,
    site_name_override: Optional[str] = None,
    overwrite: bool = True,
) -> pd.DataFrame:
    if site_folder is None and ms_folder is None:
        raise ValueError("Provide either site_folder or ms_folder.")

    if ms_folder is not None:
        ms_folder = Path(ms_folder).expanduser().resolve()
        if not ms_folder.exists():
            raise FileNotFoundError(f"MS folder does not exist: {ms_folder}")
        site_base = ms_folder.parent
    else:
        site_folder = Path(site_folder).expanduser().resolve()
        if not site_folder.exists():
            raise FileNotFoundError(f"Site folder does not exist: {site_folder}")
        ms_folder = find_ms_folder(site_folder)
        site_base = site_folder

    site_name = safe_name(site_name_override or site_base.name, keep_spaces=True)

    if output_dir is not None:
        site_output_dir = Path(output_dir).expanduser().resolve()
    else:
        if output_root is None:
            output_root = find_orthophotos_root(site_base) / "extracted VI"
        else:
            output_root = Path(output_root).expanduser().resolve()

        # Default output layout: <output_root>/<season>/<site>/MS
        # If season_name is not given, it is read from the site path
        # (for example "2024-25" or "2025-26_part1_extracted").
        inferred_season = season_name or infer_season_name_from_site_folder(site_base)
        if inferred_season:
            site_output_dir = output_root / inferred_season / site_name / "MS"
        else:
            site_output_dir = output_root / site_name / "MS"

    site_output_dir.mkdir(parents=True, exist_ok=True)

    date_folders = collect_date_folders(ms_folder, dates_to_process)
    if not date_folders:
        print(f"No date folders found under: {ms_folder}")
        return pd.DataFrame()

    print("Starting batch MS VI extraction")
    print(f"Extraction logic: {EXTRACTION_LOGIC}")
    print(f"Site: {site_name}")
    print(f"MS folder: {ms_folder}")
    print(f"Site output folder: {site_output_dir}")
    print(f"Dates selected: {len(date_folders)}")

    summary_rows = []

    for date_str, date_dir in date_folders.items():
        print("\n" + "=" * 70)
        print(f"Processing {site_name} {date_str}")

        raster_path = find_ms_tif_for_date(date_dir)
        shapefile_path = find_grid_shapefile_for_ms_date(
            ms_date_dir=date_dir,
            site_folder=site_base,
            ms_folder=ms_folder,
            plot_id_field=plot_id_field,
        )
        output_csv = site_output_dir / f"{site_name}_{date_str}_MS_VI.csv"

        row = {
            "site": site_name,
            "date": date_str,
            "date_folder": str(date_dir),
            "raster": str(raster_path) if raster_path else "",
            "shapefile": str(shapefile_path) if shapefile_path else "",
            "output_csv": str(output_csv),
            "status": "",
            "features": 0,
            "warnings": 0,
            "plot_id_field": "",
            "message": "",
        }

        if raster_path is None:
            row["status"] = "skipped_missing_ms_tif"
            row["message"] = "No MS .tif file found directly inside the date folder."
            print(f"[SKIP] {row['message']}")
            summary_rows.append(row)
            continue

        if shapefile_path is None:
            row["status"] = "skipped_missing_shapefile"
            row["message"] = "No matching RGB .shp file found for this MS date."
            print(f"[SKIP] {row['message']}")
            summary_rows.append(row)
            continue

        print(f"MS tif: {raster_path}")
        print(f"Shapefile: {shapefile_path}")
        print(f"Output CSV: {output_csv}")

        try:
            stats = extract_ms_indices_for_date(
                raster_path=raster_path,
                shapefile_path=shapefile_path,
                output_csv=output_csv,
                site_name=site_name,
                date_str=date_str,
                plot_id_field=plot_id_field,
                overwrite=overwrite,
            )
            row.update(stats)
            if stats["status"] == "processed":
                row["message"] = "OK"
                print(f"[DONE] Wrote {stats['features']} rows")
            else:
                row["message"] = "Existing output kept."
                print(f"[SKIP] {row['message']}")
        except Exception as exc:
            row["status"] = "failed"
            row["message"] = str(exc)
            print(f"[FAILED] {exc}")

        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    summary_csv = site_output_dir / f"{site_name}_MS_VI_batch_summary.csv"
    summary.to_csv(summary_csv, index=False)

    print("\n" + "#" * 70)
    print("FINAL RUN SUMMARY")
    print("#" * 70)
    print(summary["status"].value_counts(dropna=False).to_string())
    print(f"\nSummary CSV: {summary_csv}")

    return summary


def run_batch_jobs(batch_jobs: list[dict]) -> pd.DataFrame:
    """Run multiple sites and return one combined summary table."""
    all_summaries = []

    for job in batch_jobs:
        summary = run_batch_ms_vi(**job)
        if not summary.empty:
            all_summaries.append(summary)

    if not all_summaries:
        return pd.DataFrame()

    combined = pd.concat(all_summaries, ignore_index=True)
    return combined


# ============================================================
# USER SETTINGS
# Edit only this section when switching sites.
# ============================================================

if __name__ == "__main__":
    # Folder that contains MS/YYYY-MM-DD/ and RGB/YYYY-MM-DD/ date folders.
    site_folder = 'path/to/site_folder'

    # All CSVs for this site are written here.
    output_dir = 'path/to/output/MS'

    # None processes every date folder. Or give a list, e.g. ["2024-11-20"].
    dates_to_process = None

    summary = run_batch_ms_vi(
        site_folder=site_folder,
        dates_to_process=dates_to_process,
        output_dir=output_dir,
    )
    display(summary)
